# D3QN V3 — FINAL Kaggle Training & Learning Check (Train16 / Val4)

**Mục tiêu:** Kiểm chứng thuật toán D3QN có học ổn định và khai thác ROI thật sự hay chỉ chọn một cặp ΔQP gần như cố định. Không dùng Test30 và không chạy PPO.

**Pipeline:** Dataset SHA256 lock → pinned VVenC/VVdeC/YOLO toolchain → 15 unit tests → RGB conversion → native QG/bitstream preflight → 4 fixed-policy baselines trên Val4 → 20 episodes Train16 → Val4 greedy sau các mốc 1024/2048/3072/4096/5120 → chọn **best checkpoint bằng mean Val4 objective** → SAFE ZIP kèm training/validation logs + curves.

**V3 khác V2:** n-step=3, gamma=1, Polyak τ=0.005, learning rate 5e-5, epsilon còn 0.6 tại 5120 bước, và state mã hóa riêng ΔQP ROI/BG của action trước. Mọi kết luận cải thiện cần **kết quả train thực tế**, không dựa chỉ vào CI.

**Kaggle setup:** Add BDD100K Driving Video with Object Tracking (*.mov), Accelerator **GPU**, Internet **On**. Save Version → **Save & Run All**. Trên Kaggle, đường dẫn gốc BDD100K được tìm bằng VideoCatalog; không cần sửa path. **Không dùng checkpoint V2 để resume V3.**

In [ ]:
import sys, time, json, os, shutil, subprocess, zipfile, hashlib, traceback
from pathlib import Path
from statistics import mean

# Immutable experimental source. This SHA must be the V3 code, NOT the V2 pilot.
SOURCE_COMMIT="7b245aa27f8a8cc20049db31e5e7660a9a5bf165"
SOURCE_BRANCH='experiment/d3qn-native-mdp-v2'
SEED=20261111
REPO=Path('/kaggle/working/rl_vvc_d3qn_v3_source')
TOOLS=Path('/kaggle/working/rl_vvc_d3qn_v3_tools')
RUN=Path('/kaggle/working/rl_vvc_d3qn_v3_mini16_val4')
SAFE=Path('/kaggle/working/D3QN_V3_MINI16_VAL4_LEARNINGCHECK_SAFE.zip')
INPUT_ROOT=Path('/kaggle/input')
DEVICE='cuda:0'
MAX_HOURS=11.75
STARTED=time.time()
TARGET_RATE=120
MILESTONES=(1024,2048,3072,4096,5120)
CONTROLS=('qpa_anchor','global_minus2','roi_plus1_bg_plus2','roi_minus2_bg_zero')
PROTOCOL='native-masked-d3qn-qp/v3'

# To continue from a previous Kaggle version:
# 1) Add the previous SAFE ZIP as a Kaggle Input.
# 2) Set this variable to its actual /kaggle/input/.../*.zip path.
# 3) Use this same notebook/SOURCE_COMMIT/SEED; NO V2 checkpoint.
RESUME_SAFE_ZIP=None

def loadjson(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def atomicjson(path, obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(obj,indent=2,allow_nan=False),encoding='utf-8')
    temp.replace(path)

def live(cmd, *, cwd=None, log_file=None):
    command=list(map(str,cmd))
    print('$',' '.join(command),flush=True)
    log=None
    if log_file:
        file=Path(log_file);file.parent.mkdir(parents=True,exist_ok=True)
        log=file.open('w',encoding='utf-8')
    p=subprocess.Popen(command,cwd=str(cwd) if cwd else None,
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                       text=True,bufsize=1)
    try:
        for line in p.stdout:
            print(line,end='',flush=True)
            if log:log.write(line);log.flush()
        status=p.wait()
    finally:
        if p.poll() is None:
            p.terminate();p.wait()
        if log:log.close()
    if status:
        raise RuntimeError('Command failed; exit code='+str(status)+'; log='+str(log_file))
    return status

print('SOURCE_PIN',SOURCE_COMMIT)
print('EXPERIMENT',PROTOCOL,'TRAIN16 VAL4','TEST_USED=False')
print('CHECKPOINT_STEPS',MILESTONES)
RUN.mkdir(parents=True,exist_ok=True)
if not (REPO/'.git').exists():
    live(['git','clone','--branch',SOURCE_BRANCH,'--depth','1',
          'https://github.com/youngan2810/RL-VVC.git',REPO])
live(['git','fetch','--no-tags','--depth','1','origin',SOURCE_COMMIT],cwd=REPO)
live(['git','checkout','--detach',SOURCE_COMMIT],cwd=REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==SOURCE_COMMIT
assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],
                                    cwd=REPO,text=True).strip()
if str(REPO) not in sys.path:sys.path.insert(0,str(REPO))
from scripts.rl_rc_dot.data import VideoCatalog
catalog=VideoCatalog(INPUT_ROOT,mode='mini',seed=SEED)
TRAIN_IDS=list(catalog.selected['train']);VAL_IDS=list(catalog.selected['val'])
assert len(TRAIN_IDS)==16 and len(VAL_IDS)==4
assert len(set(TRAIN_IDS))==16 and len(set(VAL_IDS))==4
assert set(TRAIN_IDS).isdisjoint(VAL_IDS)
catalog.check_available(('train','val'))
for video in TRAIN_IDS+VAL_IDS:catalog.source(video)  # verify exact MOV SHA256
print('LOCKED_DATA_SHA256_PASS','TRAIN',len(TRAIN_IDS),'VAL',len(VAL_IDS))
print('VAL_IDS',VAL_IDS,'TEST_VIDEO_DECODES',0)

In [ ]:
import torch
assert torch.cuda.is_available(),'Please enable Kaggle GPU Accelerator.'
assert torch.cuda.device_count()>=1,'GPU not visible'
print('PYTORCH',torch.__version__,'CUDA',torch.version.cuda,'GPU_COUNT',torch.cuda.device_count())

if not (TOOLS/'TOOLCHAIN.json').is_file():
    live([sys.executable,'-m','scripts.rl_rc_dot.setup_kaggle',
          '--repo',REPO,'--source-commit',SOURCE_COMMIT,
          '--output',TOOLS,'--jobs','2','--install-system','--d3qn-only'],
          cwd=REPO,log_file=RUN/'logs'/'setup_stdout.log')
toolchain=loadjson(TOOLS/'TOOLCHAIN.json')
assert toolchain['source_commit']==SOURCE_COMMIT
for k in ('encoder','decoder','weights','yolo_repo'):
    assert Path(toolchain[k]).exists(),'Missing '+k
from scripts.rl_rc_dot.data import digest
assert toolchain['encoder_sha256']==digest(toolchain['encoder'])
assert toolchain['decoder_sha256']==digest(toolchain['decoder'])
print('PINNED_VVC_VVDEC_YOLO_TOOLCHAIN_PASS')

In [ ]:
# Mandatory tests and REAL native codec / QG / VVdeC audit (before RL).
live([sys.executable,'-m','unittest',
      'scripts.rl_rc_dot.test_d3qn_native_v2',
      'scripts.rl_rc_dot.test_d3qn_native_v3','-v'],
      cwd=REPO,log_file=RUN/'logs'/'unit_tests.log')
live([sys.executable,'-c',
      'from scripts.rl_rc_dot.pixels import verify_conversion; print(verify_conversion())'],
      cwd=REPO,log_file=RUN/'logs'/'rgb_parity.log')
gate_dir=RUN/'native_preflight32'
if not (gate_dir/'PREFLIGHT.json').is_file():
    if gate_dir.exists():
        shutil.rmtree(gate_dir)  # incomplete synthetic fixture only, no training data
    live([sys.executable,'-m','scripts.rl_rc_dot.preflight',
          '--encoder',toolchain['encoder'],'--decoder',toolchain['decoder'],
          '--output',gate_dir,'--frames','32','--threads','2'],
          cwd=REPO,log_file=RUN/'logs'/'native_preflight.log')
gate=loadjson(gate_dir/'PREFLIGHT.json')
assert gate['gate']=='REAL_VVC_STATE_ACTION_AU_DECODE_PASS'
assert gate['encoder_sha256']==toolchain['encoder_sha256']
print('ALL_PREFLIGHTS_PASS','NATIVE_ACTION_APPLIED','VVDEC_PARITY_PASS')

In [ ]:
import uuid

CKPT=RUN/'checkpoints';CKPT.mkdir(parents=True,exist_ok=True)
LATEST=CKPT/'latest_checkpoint.pt'
BEST=CKPT/'best_checkpoint.pt'
TRAIN_JSON=RUN/'train_history.json'
BASE_JSON=RUN/'baseline.json'
MON_JSON=RUN/'monitor_history.json'
PARTIAL_JSON=RUN/'monitor_partial.json'
MANIFEST=RUN/'learning_manifest.json'
ERROR_JSON=RUN/'last_failure.json'

if RESUME_SAFE_ZIP is not None and not MANIFEST.exists():
    resume_zip=Path(RESUME_SAFE_ZIP)
    assert resume_zip.is_file(),'Resume ZIP not found'
    with zipfile.ZipFile(resume_zip) as archive:
        allow={'train_history.json','baseline.json','monitor_history.json',
               'monitor_partial.json','learning_manifest.json',
               'checkpoints/latest_checkpoint.pt','checkpoints/best_checkpoint.pt'}
        allow.update('checkpoints/at_'+str(n)+'.pt' for n in MILESTONES)
        for key in sorted(allow.intersection(archive.namelist())):
            target=RUN/key;target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(archive.read(key))
    print('RESUME_IMPORTED',resume_zip)

def get_rows(path):
    return loadjson(path) if path.is_file() else []

training=get_rows(TRAIN_JSON)
baselines=get_rows(BASE_JSON)
monitors=get_rows(MON_JSON)
partial=get_rows(PARTIAL_JSON)
if MANIFEST.is_file():
    m=loadjson(MANIFEST)
    assert (m['source_commit'],m['seed'],m['protocol'])==(SOURCE_COMMIT,SEED,PROTOCOL)
    assert m['test_used'] is False and m['train_video_count']==16 and m['validation_video_count']==4
if training:
    assert all(r['total_steps']==256*(i+1) for i,r in enumerate(training))
    assert LATEST.is_file(),'Missing V3 checkpoint needed for resume'
    assert training[-1]['total_steps']<=5120
if monitors:
    assert all(m['steps'] in MILESTONES for m in monitors)
    assert len({m['steps'] for m in monitors})==len(monitors)
assert all(x['video'] in VAL_IDS and x['method'] in CONTROLS for x in baselines)

def best_point():
    if not monitors:return None
    return max(monitors,key=lambda m:(m['mean_objective'],-m['steps']))

def save_progress(status='partial'):
    atomicjson(TRAIN_JSON,training)
    atomicjson(BASE_JSON,baselines)
    atomicjson(MON_JSON,monitors)
    atomicjson(PARTIAL_JSON,partial)
    best=best_point()
    completed=training[-1]['total_steps'] if training else 0
    atomicjson(MANIFEST,{
        'schema':'d3qn-v3-mini16-val4-learning-check/2',
        'protocol':PROTOCOL,'source_commit':SOURCE_COMMIT,'seed':SEED,
        'train_video_count':16,'validation_video_count':4,'test_used':False,
        'monitor_rate_kbps':TARGET_RATE,'train_steps':completed,
        'completed_steps':completed,'completed_episodes':len(training),
        'baseline_points':len(baselines),
        'finished_milestones':[m['steps'] for m in monitors],
        'partial_monitor_points':len(partial),
        'best_steps':best['steps'] if best else None,
        'best_objective':best['mean_objective'] if best else None,
        'status':status})

def export_safe(status='partial'):
    save_progress(status)
    temp=SAFE.with_suffix('.zip.tmp')
    keep=[TRAIN_JSON,BASE_JSON,MON_JSON,PARTIAL_JSON,MANIFEST,ERROR_JSON,
          LATEST,BEST,gate_dir/'PREFLIGHT.json']
    keep.extend(sorted(CKPT.glob('at_*.pt')))
    keep.extend(sorted((RUN/'logs').glob('*.log')))
    keep.extend([RUN/'val4_f1_curve.png',RUN/'val4_objective_curve.png',
                 RUN/'val4_action_concentration.png',RUN/'val4_objective_by_video.csv'])
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as out:
        for path in keep:
            if path.is_file():
                out.write(path,path.relative_to(RUN).as_posix())
        out.writestr('TOOLCHAIN_FINGERPRINT.json',json.dumps({
            'source_commit':SOURCE_COMMIT,'encoder_sha256':toolchain['encoder_sha256'],
            'decoder_sha256':toolchain['decoder_sha256']},indent=2))
    temp.replace(SAFE)
    print('SAFE_CHECKPOINT',status,'steps',training[-1]['total_steps'] if training else 0,
          'baselines',len(baselines),'Val_milestones',len(monitors),
          'ZIP_MB',round(SAFE.stat().st_size/1048576,1),flush=True)
    return SAFE

def budget_allows_new_stage():
    return time.time() < STARTED + MAX_HOURS*3600 - 20*60

def args_for(phase,folder,*,checkpoint=None,train_steps=1024,methods=None,val_index=0):
    args=[sys.executable,'-u','-m','scripts.rl_rc_dot.d3qn_native_runner_v3',
          '--phase',phase,'--input-root',INPUT_ROOT,
          '--encoder',toolchain['encoder'],'--decoder',toolchain['decoder'],
          '--yolo-repo',toolchain['yolo_repo'],'--weights',toolchain['weights'],
          '--output',folder,'--device',DEVICE,'--seed',str(SEED),
          '--encoder-threads','2','--train-steps',str(train_steps),
          '--val-videos','1','--val-start',str(val_index),
          '--val-rates',str(TARGET_RATE),
          '--n-step','3','--target-tau','0.005','--learning-rate','0.00005',
          '--epsilon-floor','0.20','--epsilon-decay-steps','10240',
          '--max-hours',str(MAX_HOURS),'--session-start-utc',str(STARTED)]
    if checkpoint is not None:args+=['--checkpoint-in',str(checkpoint)]
    if methods:args+=['--val-methods',methods]
    return args

def launch(stage,phase,*,checkpoint=None,train_steps=1024,methods=None,val_index=0):
    attempt=RUN/'attempts'/(stage+'_'+uuid.uuid4().hex[:8])
    attempt.parent.mkdir(parents=True,exist_ok=True)
    log=RUN/'logs'/(stage+'_'+attempt.name[-8:]+'.log')
    live(args_for(phase,attempt,checkpoint=checkpoint,train_steps=train_steps,
                  methods=methods,val_index=val_index),cwd=REPO,log_file=log)
    result=loadjson(attempt/'result.json')
    return attempt,result

export_safe()
print('RESUME_STATE',len(training),'episodes,',len(baselines),'baseline points,',
      len(monitors),'monitored milestones')

In [ ]:
# Baselines are measured once on each locked Val video, with separate resumes.
# A stage returning partial does NOT mark the video as evaluated.
stopped=False
try:
    for index,vid in enumerate(VAL_IDS):
        existing=[x for x in baselines if x['video']==vid]
        if existing:
            assert len(existing)==len(CONTROLS) and {x['method'] for x in existing}==set(CONTROLS)
            print('BASELINE_ALREADY_DONE',index,vid)
            continue
        if not budget_allows_new_stage():
            stopped=True;print('CLOCK_SOFT_STOP_BEFORE_BASELINE',vid);break
        folder,status=launch('baseline_'+str(index),'probe',val_index=index,
                             methods=','.join(CONTROLS))
        if status['status']!='probe_complete':
            stopped=True
            print('BASELINE_SOFT_STOP',vid,status['status'])
            export_safe('baseline_partial')
            break
        rows=loadjson(folder/'validation.json')
        assert len(rows)==len(CONTROLS) and {x['method'] for x in rows}==set(CONTROLS)
        assert all(x['video']==vid and x['rate_kbps']==TARGET_RATE and x['decoder_parity'] for x in rows)
        baselines.extend(rows)
        export_safe('baseline_checkpoint')
        print('BASELINE_VIDEO_PASS',index,vid)
finally:
    export_safe('baseline_ready' if len(baselines)==16 else 'baseline_partial')

if len(baselines)==16:
    for method in CONTROLS:
        sr=[b['summary'] for b in baselines if b['method']==method]
        print('BASELINE',method,'F1',round(mean(q['mean_frame_f1'] for q in sr),6),
              'J',round(mean(q['objective'] for q in sr),6),
              'kbps',round(mean(q['actual_kbps'] for q in sr),3))
else:
    print('BASELINE_INCOMPLETE',len(baselines),'/16; resume with SAFE ZIP, do not start RL')

In [ ]:
# 20 × 256 transitions. Every 1024 train transitions: 4 independent Val videos.
# Resume handles: half-trained milestone, half-validated Val4, and interrupted baseline.
def score_monitor(rows, milestone):
    assert len(rows)==4 and {x['video'] for x in rows}==set(VAL_IDS)
    assert all(x['method']=='trained_d3qn' and x['rate_kbps']==TARGET_RATE
               and x['decoder_parity'] for x in rows)
    hist=[sum(x['summary']['actions'][a] for x in rows) for a in range(25)]
    assert sum(hist)==1024
    roi=sum(x['summary']['roi_policy_diagnostics']['roi_present_frames'] for x in rows)
    no_roi=sum(x['summary']['roi_policy_diagnostics']['roi_absent_frames'] for x in rows)
    assert roi+no_roi==1024
    return {'steps':milestone,'results':rows,
            'mean_f1':mean(x['summary']['mean_frame_f1'] for x in rows),
            'mean_objective':mean(x['summary']['objective'] for x in rows),
            'mean_actual_kbps':mean(x['summary']['actual_kbps'] for x in rows),
            'action_histogram':hist,'dominant_action':max(range(25),key=lambda k:hist[k]),
            'dominant_fraction':max(hist)/1024,
            'roi_present_frames':roi,'roi_absent_frames':no_roi}

try:
    if len(baselines)!=16:
        print('SKIPPING_TRAINING_UNTIL_ALL_BASELINE_VIDEOS_PASS')
    else:
        for milestone in MILESTONES:
            if any(x['steps']==milestone for x in monitors):
                print('MILESTONE_ALREADY_DONE',milestone)
                continue
            completed=training[-1]['total_steps'] if training else 0
            assert completed<=milestone, 'Training advanced past an unvalidated milestone'
            checkpoint_at=CKPT/('at_'+str(milestone)+'.pt')
            if completed<milestone:
                if not budget_allows_new_stage():
                    stopped=True;print('CLOCK_SOFT_STOP_BEFORE_TRAIN',milestone);break
                folder,status=launch('train_to_'+str(milestone),'train',
                                     checkpoint=LATEST if completed else None,
                                     train_steps=milestone-completed)
                rows=loadjson(folder/'train_metrics.json') if (folder/'train_metrics.json').is_file() else []
                if rows:
                    assert rows[0]['total_steps']==completed+256
                    assert all(r['total_steps']==completed+256*(i+1) for i,r in enumerate(rows))
                    training.extend(rows)
                    assert training[-1]['total_steps']<=milestone
                    shutil.copyfile(folder/'checkpoint.pt',LATEST)
                    if training[-1]['total_steps']==milestone:
                        shutil.copyfile(LATEST,checkpoint_at)
                export_safe('train_checkpoint')
                if status['status']!='train_complete' or training[-1]['total_steps']!=milestone:
                    stopped=True
                    print('TRAIN_SOFT_STOP: saved an episode-boundary V3 checkpoint',milestone)
                    break
            assert checkpoint_at.is_file(),'Missing milestone checkpoint for Val4'
            for index,vid in enumerate(VAL_IDS):
                previous=[x for x in partial if x['steps']==milestone and x['video']==vid]
                if previous:
                    assert len(previous)==1
                    print('MONITOR_VIDEO_ALREADY_DONE',milestone,vid)
                    continue
                if not budget_allows_new_stage():
                    stopped=True;print('CLOCK_SOFT_STOP_BEFORE_VALIDATION',milestone,vid);break
                folder,status=launch('monitor_'+str(milestone)+'_'+str(index),
                                     'validate',checkpoint=checkpoint_at,
                                     val_index=index,methods='trained_d3qn')
                if status['status']!='validation_complete':
                    stopped=True;print('MONITOR_SOFT_STOP',milestone,vid)
                    export_safe('monitor_partial')
                    break
                rows=loadjson(folder/'validation.json')
                assert len(rows)==1 and rows[0]['video']==vid and rows[0]['decoder_parity']
                partial.append({'steps':milestone,**rows[0]})
                export_safe('monitor_video_checkpoint')
            if stopped:break
            rows=[{k:v for k,v in x.items() if k!='steps'}
                  for x in partial if x['steps']==milestone]
            point=score_monitor(rows,milestone)
            old_best=best_point()
            if old_best is None or point['mean_objective']>old_best['mean_objective']:
                shutil.copyfile(checkpoint_at,BEST)
            monitors.append(point)
            partial[:]=[x for x in partial if x['steps']!=milestone]
            export_safe('monitor_milestone_checkpoint')
            print('VAL4_MILESTONE_PASS',milestone,
                  'F1',round(point['mean_f1'],6),
                  'J',round(point['mean_objective'],6),
                  'kbps',round(point['mean_actual_kbps'],3),
                  'DOMINANT_ACTION',point['dominant_action'],
                  'SHARE',round(point['dominant_fraction'],4),
                  'BEST_STEPS',best_point()['steps'])
except Exception as exc:
    atomicjson(ERROR_JSON,{'error_type':type(exc).__name__,'error':str(exc),
                           'traceback':traceback.format_exc()[-12000:]})
    export_safe('error')
    raise
finally:
    export_safe('complete' if len(monitors)==5 and len(training)==20 else 'partial')

print('LEARNING_CHECK_END','EPISODES',len(training),
      'VAL_MILESTONES',len(monitors),'BEST',best_point()['steps'] if best_point() else None)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

print('TRAIN16_COVERAGE',len({r['train_video'] for r in training}))
if len(training)>=16:
    assert set(r['train_video'] for r in training[:16])==set(TRAIN_IDS)
assert len({x['video'] for x in baselines})<=4
assert all(r['total_steps']==(i+1)*256 for i,r in enumerate(training))
assert set(m['steps'] for m in monitors).issubset(MILESTONES)

if monitors and len(baselines)==16:
    xs=[m['steps'] for m in monitors]
    entries=[('qpa_anchor','QPA'),('global_minus2','Global -2'),
             ('roi_plus1_bg_plus2','Fixed (+1,+2)'),
             ('roi_minus2_bg_zero','Fixed (-2,0)')]
    for axis,file,metric,title,multiplier in [
        ('F1','val4_f1_curve.png','mean_frame_f1','Val4 pseudo-detection F1',100),
        ('objective','val4_objective_curve.png','objective','Val4 F1–bitrate objective',1)]:
        fig,ax=plt.subplots(figsize=(8.5,5))
        vals=[m['mean_f1'] if axis=='F1' else m['mean_objective'] for m in monitors]
        ax.plot(xs,[multiplier*v for v in vals],'o-',label='D3QN V3 greedy')
        for method,label in entries:
            baseline=[x['summary'][metric] for x in baselines if x['method']==method]
            ax.axhline(multiplier*mean(baseline),linestyle='--',label=label)
        ax.grid(alpha=.25);ax.legend(fontsize=8)
        ax.set_xlabel('Training transitions')
        ax.set_ylabel('F1 (%)' if axis=='F1' else 'Objective J')
        ax.set_title(title+' @ 120kbps target')
        fig.tight_layout();fig.savefig(RUN/file,dpi=150);plt.show();plt.close(fig)
    fig,ax=plt.subplots(figsize=(8.5,4))
    ax.plot(xs,[100*m['dominant_fraction'] for m in monitors],'o-')
    ax.set(xlabel='Training transitions',ylabel='Most frequent greedy action (%)',
           ylim=(0,105),title='Policy collapse / action concentration on Val4')
    ax.grid(alpha=.25);fig.tight_layout()
    fig.savefig(RUN/'val4_action_concentration.png',dpi=150)
    plt.show();plt.close(fig)

    with (RUN/'val4_objective_by_video.csv').open('w',newline='') as stream:
        writer=csv.writer(stream)
        writer.writerow(['steps','video','method','pseudo_f1','objective','actual_kbps'])
        for m in monitors:
            for r in m['results']:
                writer.writerow([m['steps'],r['video'],r['method'],
                                 r['summary']['mean_frame_f1'],r['summary']['objective'],
                                 r['summary']['actual_kbps']])
        for b in baselines:
            writer.writerow([0,b['video'],b['method'],b['summary']['mean_frame_f1'],
                             b['summary']['objective'],b['summary']['actual_kbps']])
    print('MONITOR_TABLE')
    for m in monitors:
        print(m['steps'],
              'F1',round(m['mean_f1'],6),
              'J',round(m['mean_objective'],6),
              'kbps',round(m['mean_actual_kbps'],3),
              'action',m['dominant_action'],
              'dominance',round(m['dominant_fraction']*100,2),'%',
              'ROI_FRAMES',m['roi_present_frames'],
              'NO_ROI_FRAMES',m['roi_absent_frames'])

    if len(monitors)==5:
        champion=best_point()
        assert BEST.is_file()
        print('BEST_VALIDATION_CHECKPOINT',champion['steps'],
              'objective',champion['mean_objective'],
              'F1',champion['mean_f1'])
else:
    print('Not enough completed Val4 checkpoints to draw curves yet.')
export_safe('complete' if len(monitors)==5 and len(training)==20 else 'partial')

In [ ]:
from IPython.display import FileLink,display
final_status=loadjson(MANIFEST)
assert SAFE.is_file()
print('FINAL_RESULT',json.dumps(final_status,indent=2))
print('SAFE_ZIP',SAFE)
print('ZIP_CONTENTS',zipfile.ZipFile(SAFE).namelist())
os.chdir('/kaggle/working')
display(FileLink(SAFE.name))
print('Nếu Kaggle đã dừng giữa chừng, tải SAFE ZIP và resume trong Kaggle session mới.')

## Đọc kết quả và quyết định bước tiếp theo

- **Đã đủ Train16?** Manifest phải ghi 20 completed episodes, 5120 steps, và cell diagnostics xác nhận 16 video khác nhau trong 16 episode đầu.
- **Đã đủ Val4?** Năm checkpoint phải có kết quả greedy trên bốn video, cùng target 120 kbps; Test30 vẫn không dùng.
- **Có học thật?** So sánh mean objective, frame pseudo-F1, **bitrate thực** với QPA/global−2/fixed(+1,+2)/fixed(−2,0) và xem action concentration. Nếu D3QN không thắng action 19 cố định thì chưa chứng minh lợi ích chọn action theo state.
- **Tối ưu checkpoint:** Best theo objective Val4, không theo loss training hoặc số bước lớn nhất.
- **Tính khoa học:** Đây chỉ là validation trong giai đoạn phát triển; chưa phải bitrate-matched BD-rate, chưa phải BDD100K ground-truth mAP, chưa công bố kết luận thắng baseline.
- **Resume:** Download ZIP `D3QN_V3_MINI16_VAL4_LEARNINGCHECK_SAFE.zip`, gắn làm Kaggle Input, đặt đúng đường dẫn vào `RESUME_SAFE_ZIP`; giữ nguyên source commit và seed, chạy lại từ đầu notebook. Không dùng checkpoint PPO/V2.

**Nếu một cell lỗi:** ZIP SAFE gần nhất thường chứa checkpoint, manifest, log. Không xóa dữ liệu để chạy lại ngẫu nhiên: xem `last_failure.json` và log của stage đó.